# Qwen3.5-35B-A3B Uncensored - Kaggle GPU

Kaggle T4 x2 (30GB VRAM) で Qwen3.5-35B-A3B を動かすノートブック。

**設定:** Settings > Accelerator > GPU T4 x2 を選択してから実行してください。

In [ ]:
# 1. 依存関係インストール
!pip install -q llama-cpp-python huggingface-hub

In [ ]:
# 2. GPU確認
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GB")

In [ ]:
# 3. モデルダウンロード (Q4_K_M ~20GB)
from huggingface_hub import hf_hub_download

REPO_ID = "HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive"
FILENAME = "Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(f"Model downloaded: {model_path}")

In [ ]:
# 4. モデル読み込み
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_gpu_layers=-1,  # 全レイヤーGPUオフロード
    verbose=False,
)
print("Model loaded!")

In [ ]:
# 5. チャット関数
from IPython.display import Markdown, display

messages = []

def chat(user_input: str, system: str | None = None, preset: str = "chat"):
    """メッセージを送信して応答を得る"""
    presets = {
        "creative": {"temperature": 1.0, "top_p": 0.95, "top_k": 20, "repeat_penalty": 1.5},
        "precise": {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "repeat_penalty": 1.0},
        "chat": {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "repeat_penalty": 1.5},
    }
    params = presets.get(preset, presets["chat"])

    if system and not any(m["role"] == "system" for m in messages):
        messages.insert(0, {"role": "system", "content": system})

    messages.append({"role": "user", "content": user_input})

    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=2048,
        **params,
    )

    reply = response["choices"][0]["message"]["content"]
    messages.append({"role": "assistant", "content": reply})
    display(Markdown(reply))
    return reply

def clear():
    """会話履歴をリセット"""
    messages.clear()
    print("会話をリセットしました")

print("準備完了！ chat('メッセージ') で会話できます")

In [ ]:
# 6. 使ってみる
chat("こんにちは！自己紹介してください。")

In [ ]:
# 続きの会話
chat("あなたの得意なことは？")

In [ ]:
# システムプロンプト付き (例)
clear()
chat(
    "こんにちは",
    system="あなたは創造的なストーリーテラーです。ユーザーのリクエストに応じて物語やシナリオを作成します。",
    preset="creative",
)